# Basic Modeling for Seizure Detection

## Loading in data, filtering for patients with positive samples

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [1]:
import os

# os.chdir("./drive/MyDrive/MIT/6.S985/ds005873")

os.chdir("/Users/nataliebarnouw/Desktop/multimodal-seizure-detection/")
!pwd

/Users/nataliebarnouw/Desktop/multimodal-seizure-detection


In [2]:
# list_windows = os.listdir('./processed_windows')
list_windows_2sec_seizure_only = os.listdir('./processed_windows_2sec_seizure_only')

# data_windows = [l for l in list_windows if l.endswith('.h5')]
data_windows_2sec_seizure_only = [l for l in list_windows_2sec_seizure_only if l.endswith('.h5')]

In [3]:
# 2 sec version on nat's computer

with open(r'./all_eeg_paths.csv', 'r') as f:
    all_eeg_paths = f.read().split(',')
with open(r'./all_ecg_paths.csv', 'r') as r:
  all_ecg_paths = r.read().split(',')
# all_eeg_paths[:5], all_ecg_paths[:5]

In [ ]:
with open(r'/content/drive/MyDrive/MIT/6.S985/ds005873/all_eeg_paths.csv', 'r') as f:
  all_eeg_paths = f.read().split(',')

with open(r'/content/drive/MyDrive/MIT/6.S985/ds005873/all_ecg_paths.csv', 'r') as f:
  all_ecg_paths = f.read().split(',')


In [31]:
# print(len(data_windows))
print(len(data_windows_2sec_seizure_only))
print(data_windows_2sec_seizure_only)

165
['sub-012_ses-01_task-szMonitoring_run-03.h5', 'sub-092_ses-01_task-szMonitoring_run-50.h5', 'sub-022_ses-01_task-szMonitoring_run-06.h5', 'sub-073_ses-01_task-szMonitoring_run-52.h5', 'sub-043_ses-01_task-szMonitoring_run-06.h5', 'sub-073_ses-01_task-szMonitoring_run-37.h5', 'sub-011_ses-01_task-szMonitoring_run-05.h5', 'sub-079_ses-01_task-szMonitoring_run-09.h5', 'sub-073_ses-01_task-szMonitoring_run-27.h5', 'sub-073_ses-01_task-szMonitoring_run-42.h5', 'sub-035_ses-01_task-szMonitoring_run-11.h5', 'sub-092_ses-01_task-szMonitoring_run-74.h5', 'sub-115_ses-01_task-szMonitoring_run-09.h5', 'sub-091_ses-01_task-szMonitoring_run-03.h5', 'sub-101_ses-01_task-szMonitoring_run-08.h5', 'sub-062_ses-01_task-szMonitoring_run-01.h5', 'sub-073_ses-01_task-szMonitoring_run-46.h5', 'sub-113_ses-01_task-szMonitoring_run-08.h5', 'sub-121_ses-01_task-szMonitoring_run-64.h5', 'sub-092_ses-01_task-szMonitoring_run-44.h5', 'sub-022_ses-01_task-szMonitoring_run-03.h5', 'sub-035_ses-01_task-szMonito

In [ ]:
import h5py
import mne
import pandas as pd
from tqdm import tqdm
import pyarrow

data_windows = data_windows_2sec_seizure_only
data_dict = {}

for path in tqdm(data_windows):

  output_prefix = "/content/drive/MyDrive/MIT/6.S985/ds005873/processed_windows/" + path.split('.')[0] #rohan's 
  output_prefix = "/Users/nataliebarnouw/Desktop/multimodal-seizure-detection/processed_windows_2sec_seizure_only/" + path.split('.')[0] # natalie's, end part is sub-012_ses-01_task-szMonitoring_run-03
  h5_path = output_prefix + ".h5" # something like 'sub-012_ses-01_task-szMonitoring_run-03'
  meta_path = output_prefix + "_metadata.parquet"
  info_path = output_prefix + "_info.json"

  # h5py_file = h5py.File(h5_path, 'r')

  # Example: load ECG windows
  # with h5py.File(h5_path, 'r') as f:
  #   if "ecg_windows" in f:
  #     ecg = f["ecg_windows"][:]      # ← this loads as a NumPy array
  #     # print("ECG shape:", ecg.shape)

  #   # Example: load EEG windows
  #   if "eeg_windows" in f:
  #     eeg = f["eeg_windows"][:]
  #     # print("EEG shape:", eeg.shape)

  with open(meta_path, 'rb') as f:
    meta = pd.read_parquet(f)

  if meta['binary_label'].sum() > 0:
    data_dict[meta['run_id'].iloc[0]] = {
        'num_pos': meta['binary_label'].sum(), # number positive seizure windows
        'size': len(meta['binary_label']), # number total windows 
        'patient_id': meta['patient_id'].iloc[0], # patient id, like sub-001
        'prefix': output_prefix
    }


100%|██████████| 165/165 [00:01<00:00, 111.30it/s]


In [9]:
import pandas as pd

df = pd.DataFrame(data_dict).T

In [10]:
df['num_pos'].sum()

np.int64(22044)

In [16]:
df.to_csv('./patients_w_pos_samples_2sec.csv')

In [30]:
import pandas as pd

# df = pd.read_csv("./patients_w_pos_samples.csv")
df = pd.read_csv("./patients_w_pos_samples_2sec.csv")
df.head

<bound method NDFrame.head of      Unnamed: 0.1      Unnamed: 0  num_pos  size patient_id  \
0               0  sub-012_run-03      175  1050    sub-012   
1               1  sub-092_run-50       57   342    sub-092   
2               2  sub-022_run-06      110   660    sub-022   
3               3  sub-073_run-52      116   696    sub-073   
4               4  sub-043_run-06       19   114    sub-043   
..            ...             ...      ...   ...        ...   
160           160  sub-062_run-02       28   168    sub-062   
161           161  sub-073_run-20       83   498    sub-073   
162           162  sub-071_run-18       98   588    sub-071   
163           163  sub-114_run-10        5    30    sub-114   
164           164  sub-070_run-12       96   576    sub-070   

                                                prefix  
0    /Users/nataliebarnouw/Desktop/multimodal-seizu...  
1    /Users/nataliebarnouw/Desktop/multimodal-seizu...  
2    /Users/nataliebarnouw/Desktop/multimo

In [62]:
from sklearn.model_selection import train_test_split

all_patients = df['patient_id'].unique()
all_patients = [patient.split('-')[1] for patient in all_patients]
print(all_patients)
# all_patients=range(0,48)

train_patients_2, test_patients_2 = train_test_split(all_patients, test_size=0.2, train_size=0.8, random_state=42)
train_patients_2, val_patients_2 = train_test_split(train_patients_2, test_size=0.2, train_size=0.8, random_state=42)

train_patients_2 = ['-'.join(['sub', patient]) for patient in train_patients_2]
val_patients_2 = ['-'.join(['sub', patient]) for patient in val_patients_2]
test_patients_2 = ['-'.join(['sub', patient]) for patient in test_patients_2]

print(train_patients_2)
print(val_patients_2)
print(test_patients_2)

['012', '092', '022', '073', '043', '011', '079', '035', '115', '091', '101', '062', '113', '121', '100', '114', '075', '030', '119', '070', '059', '083', '071', '018', '074', '123', '090', '025', '020', '089', '032', '118', '065', '007', '107', '044', '013', '053', '094', '109', '050', '093', '036', '106', '088', '110', '005', '006']
['sub-013', 'sub-118', 'sub-011', 'sub-114', 'sub-032', 'sub-065', 'sub-107', 'sub-091', 'sub-059', 'sub-115', 'sub-044', 'sub-121', 'sub-089', 'sub-073', 'sub-094', 'sub-088', 'sub-079', 'sub-071', 'sub-100', 'sub-109', 'sub-036', 'sub-022', 'sub-006', 'sub-075', 'sub-083', 'sub-062', 'sub-092', 'sub-110', 'sub-012', 'sub-101']
['sub-035', 'sub-020', 'sub-007', 'sub-005', 'sub-119', 'sub-018', 'sub-030', 'sub-093']
['sub-025', 'sub-050', 'sub-090', 'sub-106', 'sub-074', 'sub-053', 'sub-113', 'sub-070', 'sub-043', 'sub-123']


In [63]:
df['size'].sum()

np.int64(129888)

In [65]:
train_ids = df[df['patient_id'].isin(train_patients_2)]
val_ids = df[df['patient_id'].isin(val_patients_2)]
test_ids = df[df['patient_id'].isin(test_patients_2)]

In [66]:
train_ids['size']

0      1050
1       342
2       660
3       696
5      2592
       ... 
159    1812
160     168
161     498
162     588
163      30
Name: size, Length: 114, dtype: int64

In [67]:
import os
import h5py
import pyarrow.parquet as pq
import numpy as np
import torch
from tqdm import tqdm

# INPUT_DIR = "/content/drive/MyDrive/MIT/6.S985/ds005873/processed_windows"
# OUTPUT_DIR = "/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated"

INPUT_DIR = '/Users/nataliebarnouw/Desktop/multimodal-seizure-detection/processed_windows_2sec_seizure_only'
OUTPUT_DIR = '/Users/nataliebarnouw/Desktop/multimodal-seizure-detection/consolidated_2sec'

POS_NEG_RATIO = 4  # 1 positive : 4 negative

def process_dataset_by_split(bases, split):
    eeg_list = []
    ecg_list = []

    # store all labels
    binary_label_list = []
    lateralization_list = []
    label_list = []
    localization_list = []
    vigilance_list = []
    duration_list = []

    for base in tqdm(bases):
        h5_path = os.path.join(INPUT_DIR, base + ".h5")
        meta_path = os.path.join(INPUT_DIR, base + "_metadata.parquet")

        # load metadata
        meta_df = pq.read_table(meta_path).to_pandas()

        labels = meta_df["binary_label"].to_numpy()
        lateralization = meta_df["lateralization"].to_numpy()
        other_label = meta_df["label"].to_numpy()  # assuming this is the "label" column
        localization = meta_df["localization"].to_numpy()
        vigilance = meta_df["vigilance"].to_numpy()
        duration = meta_df["seizure_duration_sec"].to_numpy(dtype=np.float32)

        # indices of positives and negatives
        pos_idx = np.where(labels == 1)[0]
        neg_idx = np.where(labels == 0)[0]

        # sample negatives according to ratio
        n_neg_sample = min(len(neg_idx), len(pos_idx) * POS_NEG_RATIO)
        neg_sample_idx = np.random.choice(neg_idx, size=n_neg_sample, replace=False)

        # combine
        selected_idx = np.concatenate([pos_idx, neg_sample_idx])
        np.random.shuffle(selected_idx)

        with h5py.File(h5_path, "r") as f:
            if "eeg_windows" in f:
                eeg = np.take(f["eeg_windows"], selected_idx, axis=0).astype(np.float32)
                eeg_list.append(eeg)

            if "ecg_windows" in f:
                ecg = np.take(f["ecg_windows"], selected_idx, axis=0).astype(np.float32)
                ecg_list.append(ecg)

        # append sampled labels
        binary_label_list.append(labels[selected_idx])
        lateralization_list.append(lateralization[selected_idx].astype(object))
        label_list.append(other_label[selected_idx].astype(object))
        localization_list.append(localization[selected_idx].astype(object))
        vigilance_list.append(vigilance[selected_idx].astype(object))
        duration_list.append(duration[selected_idx])

    # concatenate all files
    eeg_np = np.concatenate(eeg_list, axis=0)
    ecg_np = np.concatenate(ecg_list, axis=0)

    binary_labels_np = np.concatenate(binary_label_list, axis=0)
    lateralization_np = np.concatenate(lateralization_list, axis=0)
    other_label_np = np.concatenate(label_list, axis=0)
    localization_np = np.concatenate(localization_list, axis=0)
    vigilance_np = np.concatenate(vigilance_list, axis=0)
    duration_np = np.concatenate(duration_list, axis=0)

    print("Final shapes:")
    print("EEG:", eeg_np.shape, "ECG:", ecg_np.shape, "Binary labels:", binary_labels_np.shape)

    # create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # save all arrays as compressed npz
    output_path = os.path.join(OUTPUT_DIR, f"{split}_data.npz")
    np.savez_compressed(
        output_path,
        eeg=eeg_np,
        ecg=ecg_np,
        binary_label=binary_labels_np,
        lateralization=lateralization_np,
        label=other_label_np,
        localization=localization_np,
        vigilance=vigilance_np,
        seizure_duration_sec=duration_np
    )

    print(f"Saved to {output_path}")

In [68]:
process_dataset_by_split(train_ids['prefix'], 'train')
process_dataset_by_split(val_ids['prefix'], 'val')
process_dataset_by_split(test_ids['prefix'], 'test')

100%|██████████| 114/114 [00:02<00:00, 39.19it/s]


Final shapes:
EEG: (78400, 2, 512) ECG: (78400, 1, 512) Binary labels: (78400,)
Saved to /Users/nataliebarnouw/Desktop/multimodal-seizure-detection/consolidated_2sec/train_data.npz


100%|██████████| 21/21 [00:00<00:00, 48.79it/s]


Final shapes:
EEG: (12270, 2, 512) ECG: (12270, 1, 512) Binary labels: (12270,)
Saved to /Users/nataliebarnouw/Desktop/multimodal-seizure-detection/consolidated_2sec/val_data.npz


100%|██████████| 30/30 [00:00<00:00, 46.04it/s]


Final shapes:
EEG: (18515, 2, 512) ECG: (18515, 1, 512) Binary labels: (18515,)
Saved to /Users/nataliebarnouw/Desktop/multimodal-seizure-detection/consolidated_2sec/test_data.npz


## Modeling starts here

In [70]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class SeizureDataset(Dataset):

  def __init__(self, data_path, eeg_transform=None, ecg_transform=None):
    data = np.load(data_path, allow_pickle=True)

    self.eeg = data['eeg']
    self.ecg = data['ecg']

    self.binary_label=data['binary_label']
    self.lateralization=data['lateralization']
    self.label=data['label']
    self.localization=data['localization']
    self.vigilance=data['vigilance']
    self.seizure_duration_sec=data['seizure_duration_sec']

    self.eeg_transform = eeg_transform
    self.ecg_transform = ecg_transform

  def __len__(self):
    return len(self.eeg)

  def __getitem__(self, idx):
    eeg = self.eeg[idx]
    ecg = self.ecg[idx]
    label = self.binary_label[idx]

    if self.eeg_transform:
      eeg = self.eeg_transform(eeg)

    if self.ecg_transform:
      ecg = self.ecg_transform(ecg)

    return ecg, eeg, label

In [71]:
trainset = SeizureDataset('./consolidated/train_data.npz')
valset = SeizureDataset('./consolidated/val_data.npz')
testset = SeizureDataset('./consolidated/test_data.npz')

In [72]:
trainloader = DataLoader(trainset, batch_size=8, shuffle=True)
valloader = DataLoader(valset, batch_size=8, shuffle=False)
testloader = DataLoader(testset, batch_size=8, shuffle=False)

In [74]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoModalityMLP(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        # Modality 1 (vector)
        self.m1 = nn.Linear(2560, hidden)

        # Modality 2 (2 × 2560 signal)
        self.conv = nn.Sequential(
            nn.Conv1d(2, 16, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)   # output → (B, 32, 1)
        )
        self.m2 = nn.Linear(32, hidden)

        # Fusion
        self.fuse = nn.Linear(2 * hidden, 1)

    def forward(self, x1, x2):
        # x1: (B, 1, dim1)

        # x2: (B, 2, 2560)
        h1 = F.relu(self.m1(x1)).squeeze()

        feat = self.conv(x2).squeeze(-1)  # (B, 32)
        h2 = F.relu(self.m2(feat))

        h = torch.cat([h1, h2], dim=1)
        return self.fuse(h)

In [77]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

def train_model(model, trainloader, valloader, epochs, lr=1e-4, device='cuda'):
    model = model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    pos_w = torch.tensor([4.0], device=device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    for ep in range(1, epochs + 1):
        model.train()
        pbar = tqdm(trainloader, desc=f"Epoch {ep}")

        for m1, m2, y in pbar:
            m1, m2, y = m1.to(device), m2.to(device), y.to(device).float()
            pred = model(m1, m2).squeeze()
            loss = bce(pred, y)

            optim.zero_grad()
            loss.backward()
            optim.step()
            # pbar.set_postfix(loss=loss.item())

        # ---- Validation AUC ----
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for m1, m2, y in valloader:
                m1, m2 = m1.to(device), m2.to(device)
                pred = model(m1, m2).squeeze().cpu()
                preds.append(pred)
                trues.append(y)

        preds = torch.cat(preds).numpy()
        trues = torch.cat(trues).numpy()
        auc = roc_auc_score(trues, preds)

        print(f"Epoch {ep} | Val AUC: {auc:.4f}")

In [78]:
model = TwoModalityMLP()

train_model(model, trainloader, valloader, epochs=10, device='cpu')

Epoch 1: 100%|██████████| 4634/4634 [00:07<00:00, 608.30it/s]


Epoch 1 | Val AUC: 0.6184


Epoch 2: 100%|██████████| 4634/4634 [00:07<00:00, 649.52it/s]


Epoch 2 | Val AUC: 0.6334


Epoch 3: 100%|██████████| 4634/4634 [00:07<00:00, 642.81it/s]


...

Epoch 8 | Val AUC: 0.6711


Epoch 9: 100%|██████████| 4634/4634 [00:08<00:00, 556.20it/s]


Epoch 9 | Val AUC: 0.6762


Epoch 10: 100%|██████████| 4634/4634 [00:07<00:00, 582.28it/s]


Epoch 10 | Val AUC: 0.6785


In [79]:
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
import torch

def test_model(model, testloader, device='cuda'):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for m1, m2, y in testloader:
            m1, m2 = m1.to(device), m2.to(device)
            logits = model(m1, m2).squeeze()
            probs = torch.sigmoid(logits).cpu()

            all_probs.append(probs)
            all_labels.append(y)

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()

    # ---- Metrics ----
    auc = roc_auc_score(labels, probs)
    preds = (probs >= 0.5).astype(int)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    specificity = tn / (tn + fp + 1e-9)
    sensitivity = tp / (tp + fn + 1e-9)

    # ---- Print summary ----
    print("==== Test Results ====")
    print(f"AUC:          {auc:.4f}")
    print(f"Accuracy:     {acc:.4f}")
    print(f"Precision:    {prec:.4f}")
    print(f"Recall:       {rec:.4f}")
    print(f"F1 Score:     {f1:.4f}")
    print(f"Sensitivity:  {sensitivity:.4f}")
    print(f"Specificity:  {specificity:.4f}")
    print(f"TP/FP/FN/TN:  {tp}/{fp}/{fn}/{tn}")

    return {
        "auc": auc,
        "acc": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

In [81]:
test_model(model, testloader, device='cpu')

==== Test Results ====
AUC:          0.6745
Accuracy:     0.7454
Precision:    0.3739
Recall:       0.4045
F1 Score:     0.3886
Sensitivity:  0.4045
Specificity:  0.8307
TP/FP/FN/TN:  919/1539/1353/7549


{'auc': 0.6745318030819158,
 'acc': 0.7454225352112676,
 'precision': 0.3738812042310822,
 'recall': 0.4044894366197183,
 'f1': 0.3885835095137421,
 'sensitivity': np.float64(0.4044894366195403),
 'specificity': np.float64(0.8306558098590635),
 'tp': np.int64(919),
 'fp': np.int64(1539),
 'fn': np.int64(1353),
 'tn': np.int64(7549)}